In [1]:
#import mysql.connector
#from mysql.connector import Error
ConnectString = "mysql -h database-1.clqxqhhe6wft.us-east-1.rds.amazonaws.com -P 3306 -u admin -p'<Enter_DB_Password>' --ssl-verify-server-cert  --ssl-ca=/certs/global-bundle.pem mysql"

host=ConnectString[9:60]
user='admin'
password='Data608-Project'
database='CHESSBOT'


In [2]:

import nextmove as nm
import metrics as mt
import movelist as ml
import boardImage as bI
import snark as sn
import pagelayout as pl
import mysql.connector
from mysql.connector import Error
from IPython.display import display, HTML
from datetime import datetime



In [3]:
#this chunk is to define all values and variables that are passed back to the program from the webpage

SourceSpaceColoum = "b"  #from a - h
SourceSpaceRow = "2"  # from 1 - 8
DestinationSpaceColoum = "b" #from a - h
DestinationSpaceRow = "3"  # from 1 - 8

BoardRow1 = "r,n,b,q,k,b,n,r";    #row 1 a through h
BoardRow2 = "p,p,p,p,0,p,p,p";
BoardRow3 = "0,0,0,0,0,0,0,0";
BoardRow4 = "0,0,0,0,p,0,0,0";
BoardRow5 = "0,0,0,0,P,0,0,0";
BoardRow6 = "0,0,0,0,0,0,0,0";
BoardRow7 = "P,P,P,P,0,P,P,P";
BoardRow8 = "R,N,B,Q,K,B,N,R";   # row 8  a through h

AllMovesString = "1. e2e3 d7d5"
# string built for current came

BotDifficulty = "1000"  #1000-Easy ,  1500 - Medium,  2500 - Hard,  5000 - Grandmaster
MetricDisplay = "1"   #1 - display metrics   0 - do not display
RemarkType = "first move"  #1) first move  2) general move  3)Black won 4) UniqueMovePosition 5)Illegal Move
SnarkLevel = "neutral" #off, neutral, positive, evil


In [4]:
#call NextMove

#Boardlayout is 8x8 array (where BoardLayout[x][y] is addressing  (row x+1) (column y+1))
BoardLayout = [BoardRow1.split(","),BoardRow2.split(","),BoardRow3.split(","),BoardRow4.split(","),BoardRow5.split(","),BoardRow6.split(","),BoardRow7.split(","),BoardRow8.split(",")]

UserMove = SourceSpaceColoum +  SourceSpaceRow +  DestinationSpaceColoum + DestinationSpaceRow

#connect to database
connection = mysql.connector.connect(
    host=host,
    user=user,
    password=password,
    database='CHESSBOT' # Optional: leave out to create DB first
   )

if connection.is_connected():
    cursor = connection.cursor()
else:
    exit()

BotMove,Metrics,BoardLayout = nm.nextmove(BoardLayout,AllMovesString,UserMove,cursor,BotDifficulty)
print('here')


1. e2e3 d7d5 2. b2b3 e7e5 3. c1b2 d8g5 4. g2g3 g8f6 5. f1h3 b8c6 6. d1f3 f8b4 7. c2c3 e8g8 8. g1e2 c8g4 9. f3g2 g4e2 10. e1e2
SELECT * FROM moves WHERE  Set5 LIKE '1. e2e3 d7d5 2. b2b3 e7e5 3. c1b2 d8g5 4. g2g3 g8f6 5. f1h3 b8c6 %' AND Set10 LIKE ' 6. d1f3 f8b4 7. c2c3 e8g8 8. g1e2 c8g4 9. f3g2 g4e2 10. e1e2 %' 
Total Games found: 0


IndexError: list index out of range

In [ ]:
#call metrics
BoardLayout = mt.metrics(Metrics, BotMove, BoardLayout)

In [ ]:
#call pagelayout
FirstPartOfPage = '<!DOCTYPE html><html><head><meta charset="UTF-8"><title>Chess Bot by Rita Group</title></head>'
FirstPartOfPage += '<body><div style="font-size: 34px; background-color: black; color: white; width: 100%; text-align: center;">'


#define Header 
FirstPartOfPage += '<h1><BIG>Welcome to Chess Bot </BIG></h1>'
FirstPartOfPage += '<table style="font-size: 24px; background-color: black; color: white; margin: 0 auto; width: 50%; border-collapse: collapse; text-align: center;"><tbody><tr><td>'
FirstPartOfPage += '<table><tbody>'

#obtain BoardLayout,MovesList
BaseBoard,BaseMovesList = pl.pagelayout()

SecondPartOfPage = """</tbody></table>
	</td><td>
<table style="font-size: 24px; width: 100%; max-width: 400px; border-collapse: collapse;">
  <thead>
    <tr style="background-color: black;">
      <th style="padding: 6px; border: 2px solid #ddd;">#</th>
      <th style="padding: 6px; border: 2px solid #ddd;">White</th>
      <th style="padding: 6px; border: 2px solid #ddd;">Black</th>
    </tr>
  </thead>
  <tbody>"""


ThirdPartOfPage = """ </tbody>
</table>
</td></tbody></table>	
<hr>
	<form method="POST"><table style="font-size: 24px; background-color: black; color: white; margin: 0 auto; width: 50%; border-collapse: collapse; text-align: center;"><tbody><tr>
	<td rowspan="2" style="font-size: 34px;  border: 2px solid #555;  background-color: black; color: white;padding-left: 50px; padding-right: 50px;">Next Move:</td>
	<td style="border: 2px solid #555;padding: 16px;">
	Source Coloum </td>
	<td style="border: 2px solid #555;padding: 16px;">
	Source Row </td>
    <td style="border: 2px solid #555;padding: 16px;">
	Destination Coloum </td>
    <td style="border: 2px solid #555;padding: 16px;">
	Destination Row </td>
		<td rowspan="2" style="font-size: 34px;  border: 2px solid #555;  background-color: black; color: white;padding-left: 50px; padding-right: 50px;"><button type="button" style="font-size: 24px; background-color: black; color: gold; border: 2px solid gold; padding: 15px 30px; cursor: pointer; border-radius: 8px; font-weight: bold; margin: 20px auto; display: block;">
  Submit
</button></td>

	</tr><tr>
	<td style="border: 2px solid #555;">
		<select id="move-select" name="moves" style="font-size: 34px; padding-left: 20px; padding-right: 20px;">
			<option value="a">A</option>
			<option value="b">B</option>
			<option value="c">C</option>
			<option value="d">D</option>
			<option value="e">E</option>
			<option value="f">F</option>
			<option value="g">G</option>
			<option value="h">H</option>
			</select>
	</td>
	<td style="border: 2px solid #555;">
		<select id="move-select" name="moves" style="font-size: 34px; padding-left: 20px; padding-right: 20px;">
			<option value="1">1</option>
			<option value="2">2</option>
			<option value="3">3</option>
			<option value="4">4</option>
			<option value="5">5</option>
			<option value="6">6</option>
			<option value="7">7</option>
			<option value="8">8</option>
			</select>
	</td>

	<td style="border: 2px solid #555;">
		<select id="move-select" name="moves" style="font-size: 34px; padding-left: 20px; padding-right: 20px;">
			<option value="a">A</option>
			<option value="b">B</option>
			<option value="c">C</option>
			<option value="d">D</option>
			<option value="e">E</option>
			<option value="f">F</option>
			<option value="g">G</option>
			<option value="h">H</option>
			</select>
	</td>
	<td style="border: 2px solid #555;">
		<select id="move-select" name="moves" style="font-size: 34px; padding-left: 20px; padding-right: 20px;">
			<option value="1">1</option>
			<option value="2">2</option>
			<option value="3">3</option>
			<option value="4">4</option>
			<option value="5">5</option>
			<option value="6">6</option>
			<option value="7">7</option>
			<option value="8">8</option>
			</select>
	</td>


	</tr></tbody></table></form>
<p style="color: red; font-style: italic;font-size: 24px;">
"""

FourthPartOfPage = """</p><br><br><br><br></div>
	
</body>
</html>
"""


In [ ]:
#call movelist

BaseMovesList = ml.movelist(UserMove, BotMove, BaseMovesList)


for row in BaseMovesList:
    SecondPartOfPage += """
<tr>"""
    SecondPartOfPage += row[0]
    SecondPartOfPage += row[1]
    SecondPartOfPage += row[2]
    SecondPartOfPage += row[3]
    SecondPartOfPage += row[4]
    SecondPartOfPage += row[5]
    SecondPartOfPage += '</tr>'



In [ ]:
#call boardimage ~ 64 times

for r_idx, row in enumerate(BaseBoard):
    FirstPartOfPage += "\n<tr>"
    
    for c_idx, space in enumerate(row):
        # Create a coordinate string, e.g., "3,1"
        space_address = f"{r_idx},{c_idx}" 
        
        FirstPartOfPage += space[0]
        # Pass the dynamic address instead of the hardcoded "space[1]"
        FirstPartOfPage += bI.boardImage(BoardLayout, space_address)
        FirstPartOfPage += space[2]
        
    FirstPartOfPage += '</tr>'


In [ ]:
#call snark remark

RemarkText = sn.snark(RemarkType,SnarkLevel,len(BaseMovesList))

FourthPartOfPage = RemarkText + FourthPartOfPage

In [ ]:
#output webpage
display(HTML(FirstPartOfPage+SecondPartOfPage+ThirdPartOfPage+FourthPartOfPage))


In [ ]:
#output webpage raw
print(FirstPartOfPage)
print(SecondPartOfPage)
print(ThirdPartOfPage)
print(FourthPartOfPage)